STEP 1

#1.) Standardize Timestamps - AirGradient often exports timestamps in UTC. If you don't localize them, your data will be offset by 8 hours, and you’ll accidentally match Manila's midnight traffic to a midday satellite overpass.

Action: Convert the timestamp column to datetime objects and shift them to Manila time (PHT, UTC+8).

Pandas Logic: df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_convert('Asia/Manila')

#2.) Eliminate Implausible Values - Low-cost laser sensors occasionally misread extreme values due to sudden electrical surges or water droplets trapping light inside the chamber.Action: Drop physically impossible rows.The Thresholds: * Remove anything where $PM_{2.5} < 0$.Remove anything where $PM_{2.5} > 500\ \mu g/m^3$ (Unless there was a massive documented fire right next to the sensor, readings this high are almost always hardware errors).

#3.) Catch "Stuck" Sensors - When a sensor loses connection or its internal script stalls, it might continue repeating the exact same decimal number over and over for hours.Action: Write a look-ahead filter to flag repeating sequences. If the $PM_{2.5}$ value does not change by even $0.01$ for more than 3 consecutive readings (especially in the 10-minute data), drop those rows.

#4.) Statistical Outlier Removal - This is the most critical step for your thesis. If someone sweeps dust directly under a sensor or smokes a cigarette next to it, the sensor will register a massive spike. This represents localized noise, not the true background ambient air quality of the grid cell that the satellite sees.

Action: Use a Rolling Z-Score or Rolling Median Absolute Deviation (MAD) window.

The Logic: Look at a moving 2-hour window. If a specific 10-minute reading is more than 3 standard deviations away from the window's median, replace it with NaN or drop it. This smooths out hyper-local anomalies without deleting a real regional pollution event.

#5.) Bring Everything to the Hourly Level - Once the 10-minute data is completely sanitized, you can safely aggregate it to match the historical 1-hour backbone data.

Group by the hour: Take the mean of your clean 10-minute blocks.

Apply the 75% Rule: Only keep the hour if it contains at least 5 out of the 6 possible 10-minute readings. If a sensor went offline for 40 minutes of that hour, discard the whole hour because it isn't representative.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve data folder from workspace root or notebook folder
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("..") / "data"

RAW_DIR = DATA_DIR / "raw"
if not RAW_DIR.exists():
    RAW_DIR = DATA_DIR

OUTPUT_DIR = DATA_DIR / "cleaned"
MIN_DIR = OUTPUT_DIR / "min-buckets"
HOUR_DIR = OUTPUT_DIR / "hour-buckets"

MIN_DIR.mkdir(parents=True, exist_ok=True)
HOUR_DIR.mkdir(parents=True, exist_ok=True)


def find_pm25_col(columns: list[str]) -> str:
    pm_cols = [c for c in columns if str(c).startswith("PM2.5")]
    if not pm_cols:
        raise ValueError("No PM2.5 column found.")

    preferred = [
        c for c in pm_cols
        if "correct" in c.lower() or c.lower().endswith(" c") or "corr" in c.lower()
    ]
    return preferred[0] if preferred else pm_cols[0]


def find_timestamp_col(columns: list[str]) -> str:
    for candidate in ["UTC Date/Time", "timestamp", "Local Date/Time"]:
        if candidate in columns:
            return candidate
    raise ValueError("No timestamp column found.")


def clean_10min_df(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    pm_col = find_pm25_col(df.columns.tolist())
    ts_col = find_timestamp_col(df.columns.tolist())

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp", pm_col]).sort_values("timestamp")

    # Step 2: remove implausible values
    df = df[(df[pm_col] >= 0) & (df[pm_col] <= 500)]

    # Step 3: drop runs of unchanged values (>= 4 consecutive readings)
    pm_round = df[pm_col].round(2)
    run_id = (pm_round != pm_round.shift()).cumsum()
    run_len = pm_round.groupby(run_id).transform("size")
    df = df[run_len < 4]

    # Step 4: rolling outlier removal (2-hour window = 12x10min)
    window = 12
    rolling_median = df[pm_col].rolling(window=window, min_periods=6, center=True).median()
    rolling_std = df[pm_col].rolling(window=window, min_periods=6, center=True).std()
    z = (df[pm_col] - rolling_median).abs() / rolling_std
    df.loc[z > 3, pm_col] = np.nan
    df = df.dropna(subset=[pm_col])

    return df, pm_col


def aggregate_hourly(df: pd.DataFrame, pm_col: str) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["timestamp"].dt.floor("H")

    counts = df.groupby("hour")[pm_col].count()
    keep_hours = counts[counts >= 5].index
    df = df[df["hour"].isin(keep_hours)]

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    agg_map = {col: "mean" for col in numeric_cols}

    non_numeric = [
        c for c in df.columns
        if c not in numeric_cols and c not in ["timestamp", "hour"]
    ]
    agg_map.update({col: "first" for col in non_numeric})

    hourly = df.groupby("hour", as_index=False).agg(agg_map)
    hourly = hourly.rename(columns={"hour": "timestamp"})
    return hourly


summary = []
for file_path in sorted(RAW_DIR.glob("*-10MIN.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, pm_col = clean_10min_df(df_raw)
    df_hourly = aggregate_hourly(df_clean, pm_col)

    clean_name = file_path.stem + "-clean.csv"
    hourly_name = file_path.stem.replace("-10MIN", "-10MIN-Agg") + ".csv"

    df_clean.to_csv(MIN_DIR / clean_name, index=False)
    df_hourly.to_csv(HOUR_DIR / hourly_name, index=False)

    summary.append(
        (file_path.name, len(df_raw), len(df_clean), len(df_hourly))
    )

summary_df = pd.DataFrame(
    summary,
    columns=["file", "raw_rows", "clean_10min_rows", "hourly_rows"],
)
summary_df

C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df[

,file,raw_rows,clean_10min_rows,hourly_rows
0,cubao-10MIN.csv,19898,19119,3142
1,lawton-10MIN.csv,18882,18533,3078
2,makati-10MIN.csv,2223,2220,367
3,pasay-10MIN.csv,4936,4733,765
4,SDG-10MIN.csv,663,654,101


STEP 2 - CLEAN HISTORICAL 1HR DATA

In [2]:
def clean_1hr_df(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    pm_col = find_pm25_col(df.columns.tolist())
    ts_col = find_timestamp_col(df.columns.tolist())

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp", pm_col]).sort_values("timestamp")

    # Remove implausible values
    df = df[(df[pm_col] >= 0) & (df[pm_col] <= 500)]

    # Drop runs of unchanged values (>= 3 consecutive hourly readings)
    pm_round = df[pm_col].round(2)
    run_id = (pm_round != pm_round.shift()).cumsum()
    run_len = pm_round.groupby(run_id).transform("size")
    df = df[run_len < 3]

    return df, pm_col


summary_1hr = []
for file_path in sorted(RAW_DIR.glob("*-1HR.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, pm_col = clean_1hr_df(df_raw)

    clean_name = file_path.stem + "-clean.csv"
    df_clean.to_csv(HOUR_DIR / clean_name, index=False)

    summary_1hr.append(
        (file_path.name, len(df_raw), len(df_clean))
    )

summary_1hr_df = pd.DataFrame(
    summary_1hr,
    columns=["file", "raw_rows", "clean_rows"],
)
summary_1hr_df

,file,raw_rows,clean_rows
0,cubao-1HR.csv,3332,3253
1,lawton-1HR.csv,3162,3141
2,makati-1HR.csv,376,376
3,pasay-1HR.csv,847,826
4,SDG-1HR.csv,127,126
